# 10a Prepare the SI Concept Corpus

Join screened pair-year PMID membership to the existing abstract cache and SI output. This notebook creates the fixed development/confirmation pair split and estimates labeling volume before any model calls.

The split is deterministic (20% confirmation, seed 42). Notebook 10b identifies articles shared by development and confirmation pairs and excludes them from concept-generation examples.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "amr").exists():
    raise RuntimeError(f"Could not find repository root from {Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from amr.pipeline_config import cfg, load_pipeline_config
from amr.pipelines.concept_notebook_workflows import prepare_concept_corpus
PIPELINE_CONFIG = load_pipeline_config(project_root=PROJECT_ROOT)
NUM_PATHOGENS = 10  # None uses all eligible pathogens; all antibiotics are retained either way.
HELDOUT_FRACTION = 0.20
SPLIT_SEED = 42


## Load and save the corpus

### What this stage builds

The screened table defines article membership for each pathogen–antibiotic pair and year. The SI table restricts analysis to pair-years with an observed outcome; the abstract cache supplies text for each PMID. A paper can belong to several pairs, so the corpus retains one row per pair-year–PMID association, while the separate abstract table keeps one text record per PMID.

Set `NUM_PATHOGENS` before running this section. An integer takes the highest-ranked eligible pathogens available in both the SI and screened data; `None` keeps the full corpus. The limit is applied before splitting pairs, and all antibiotics for selected pathogens remain included. Rebuild the corpus and split whenever this setting changes.

In [ ]:
RESULT = prepare_concept_corpus(
    PROJECT_ROOT, num_pathogens=NUM_PATHOGENS, heldout_fraction=HELDOUT_FRACTION,
    split_seed=SPLIT_SEED,
)
print({key: RESULT[key] for key in ["selected_pathogens", "pathogen_limit", "all_antibiotics_retained", "split_labels"]})
display(RESULT["preflight"])


## Jev request-volume preflight

Jev sends one typed concept-presence question per catalog entry for each unique abstract. Review the selected corpus size before enabling classification.

The request count is the number of unique nonblank abstracts after any pathogen restriction. Jev receives one request per abstract, no matter how many pairs cite it, and each request carries a question for each catalog concept. This is a request-count estimate, not a monetary cost estimate. Check the count and abstract coverage before enabling classification in notebook 10c.

In [ ]:
print({
    "Jev model": str(cfg(load_pipeline_config(project_root=PROJECT_ROOT), "system1.model", "typesafe/jev-1.13")),
    "Jev requests for full run": RESULT["preflight"]["unique_abstracts_to_classify"],
})


## Data checks

These checks verify that exploding PMID lists did not create duplicate pair-year–article links and that both split partitions contain pairs. Review abstract coverage too: missing or blank text cannot contribute a classified score. Shared PMIDs may occur across the two pair partitions, but notebook 10b removes those records from concept-generation examples.

In [ ]:
display(RESULT["coverage"])
